In [78]:
import numpy as np
import json
from classy import Class
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import cosmoprimo
import os
import pandas as pd

from FishLSS.fisherForecast import fisherForecast
from FishLSS.experiment import experiment

In [79]:
# filename for example survey
bfn = 'DESI_fullsky'
# bfn_no_subscripts = bfn.replace('_14bins', '')
# bfn_no_subscripts = bfn.replace('_28bins', '')
# bfn_no_subscripts = bfn.replace('_', '')
bd = '/home/adrien/PDM/code/PDM2026_wsl/derivatives/'

log_path = '/home/adrien/miniconda3/envs/fishlss/lib/python3.11/site-packages/FishLSS/bao_recon/log.npy'

In [80]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)

In [81]:
print('summary = ')
for keys in summary.keys():
    print(keys, summary[keys])

summary = 
Forecast name DESI_fullsky
Edges of redshift bins [0.15, 0.42, 0.6, 0.8, 1.05, 1.6]
Centers of redshift bins [0.285, 0.51, 0.7, 0.925, 1.3250000000000002]
Linear Eulerian bias in each bin [1.65, 2.0, 2.0, 2.0, 1.2]
Number density in each bin [0.0007650000000000001, 0.0004, 0.0004, 0.0003, 0.00022499999999999989]
fsky 0.75
CLASS default parameters {'output': 'tCl lCl mPk', 'non linear': 'halofit', 'l_max_scalars': 2000, 'lensing': 'yes', 'A_s': 2.083e-09, 'n_s': 0.9649, 'alpha_s': 0.0, 'h': 0.6736, 'N_ur': 2.0328, 'N_ncdm': 1, 'm_ncdm': 0.06, 'tau_reio': 0.0544, 'omega_b': 0.02237, 'omega_cdm': 0.12, 'Omega_k': 0.0, 'P_k_max_h/Mpc': 2.0, 'z_pk': '0.0,6'}


In [82]:
params = summary['CLASS default parameters']
cosmo = Class() 
cosmo.set(params) 
cosmo.compute() 

In [83]:
# load fiducial linear bias/number density from table
zs, bs, ns = np.genfromtxt('/home/adrien/PDM/code/PDM2026_wsl/FishLSS_script/' + bfn + '.txt').T

# assume zs spans the full survey
ze = summary['Edges of redshift bins']
zmin = ze[0]
zmax = ze[-1]

z_centers = summary['Centers of redshift bins']

# interpolate
b = interp1d(zs,bs)
n = interp1d(zs,ns)

nbins = len(ze)-1
fsky = summary['fsky']

exp = experiment(zedges=np.array(ze), nbins=nbins, fsky=fsky, b=b, n=n)

name = summary['Forecast name']

In [84]:
forecast = fisherForecast(experiment=exp,cosmo=cosmo,name=name,basedir=bd)

In [85]:
basis = np.array(['alpha_perp','alpha_parallel','b'])

# set recon = True, so that we perform BAO reconstruction when computing the power spectrum
forecast.recon = True

# set the "marginalized parameters", aka the derivatives, to be [alpha's, linear b]
forecast.free_params = basis

derivs = forecast.load_derivatives(basis) # load the pre computed derivatives

In [86]:
F = lambda i: forecast.gen_fisher(basis, 100, derivatives=derivs, zbins=np.array([i]))
Fs = [F(i) for i in range(nbins)]
Fs = np.array(Fs)

In [87]:
Finvs = [np.linalg.inv(Fs[i]) for i in range(nbins)]
saperp = [np.sqrt(Finvs[i][0,0]) for i in range(nbins)]
saparr = [np.sqrt(Finvs[i][1,1]) for i in range(nbins)]

### Save $\alpha_\perp$ $\alpha_\parallel$ errors

In [88]:
def create_DESI_fid_data(redshifts, DESI_style=False):
    cosmo_planck = cosmoprimo.fiducial.DESI()
    bkg = cosmo_planck.get_background(engine="class")
    thermo = cosmo_planck.get_thermodynamics()
    rdrag = thermo.rs_drag  # no parentheses needed, it's a property

    DM = bkg.comoving_angular_distance(redshifts)
    DH = 1 / bkg.efunc(redshifts) * 2997.92  # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV = (redshifts * DM**2 * DH)**(1/3)
    DM_rd = DM / rdrag
    DH_rd = DH / rdrag
    DV_rd = DV / rdrag

    data_typ = ['DH_over_rs', 'DM_over_rs', 'DV_over_rs']

    fake_data = []
    if isinstance(DESI_style, list):
        for i in range(len(redshifts)):
            if DESI_style[i]:
                line = [f"{float(redshifts[i]):.8e}", f"{DV_rd[i]:.8e}", data_typ[2]]
                fake_data.append(line)
            else:
                line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
                fake_data.append(line)
                line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
                fake_data.append(line)
    elif isinstance(DESI_style, bool) and DESI_style:
        for i in range(1, len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

        line = [f"{float(redshifts[0]):.8e}", f"{DV_rd[0]:.8e}", data_typ[2]]
        fake_data.insert(0, line)
    else:
        for i in range(len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

    return np.array(fake_data)

def save_mean_data(fake_data, folder, filename, overwrite=False):
    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, fake_data, fmt="%s")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, fake_data, fmt="%s")

def cov_from_Fisher_with_units(redshifts, Fish_inv, nbins, with_units=True, DESI_style=False):
    cov_mat = np.zeros((2*nbins, 2*nbins))
    mean = create_DESI_fid_data(redshifts, DESI_style=False)

    if not with_units:
        mean[:,1] = 1.0


    for i in range(nbins):
        if isinstance(DESI_style, list):
            if i == 0 : cov_mat = np.zeros((2*nbins-len([a for a in DESI_style if a]), 2*nbins-len([a for a in DESI_style if a])))
            if DESI_style[i]:
                sig2_DM = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                sig2_DH = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                DV = (redshifts[i] * float(mean[2*i,1])**2 * float(mean[2*i+1,1]))**(1/3)
                sig2_DV = DV**2 * ( (2/3)**2 * (sig2_DM / float(mean[2*i,1])**2) + (1/3)**2 * (sig2_DH / float(mean[2*i+1,1])**2) )
                cov_mat[i,i] = sig2_DV
            else:
                cov_mat[2*i-1,2*i-1] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                cov_mat[2*i,2*i] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                cov_mat[2*i-1,2*i] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
                cov_mat[2*i,2*i-1] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])

        elif isinstance(DESI_style, bool) and DESI_style:
            if i == 0:
                sig2_DM = Fish_inv[i][0,0] * float(mean[0,1])**2
                sig2_DH = Fish_inv[i][1,1] * float(mean[1,1])**2
                DV = (redshifts[i] * float(mean[0,1])**2 * float(mean[1,1]))**(1/3)
                sig2_DV = DV**2 * ( (2/3)**2 * (sig2_DM / float(mean[0,1])**2) + (1/3)**2 * (sig2_DH / float(mean[1,1])**2) )
                cov_mat = np.zeros((2*nbins-1, 2*nbins-1))
                cov_mat[0,0] = sig2_DV
            else:
                cov_mat[2*i-1,2*i-1] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                cov_mat[2*i,2*i] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                cov_mat[2*i-1,2*i] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
                cov_mat[2*i,2*i-1] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])

        else:
            cov_mat[2*i,2*i] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
            cov_mat[2*i+1,2*i+1] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
            cov_mat[2*i,2*i+1] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
            cov_mat[2*i+1,2*i] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])
    
    return cov_mat

def save_a_cov_mat(cov_mat, folder, filename, overwrite=False):

    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, cov_mat, fmt="%.8e")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, cov_mat, fmt="%.8e")

In [89]:
overwrite = False

saving cov mat mean values and redshifts

In [90]:
folder_to_save = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/' + bfn
if not os.path.exists(folder_to_save):
    os.makedirs(folder_to_save)

save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=False, DESI_style=False),
                folder_to_save , '/cov_alpha.txt', overwrite=overwrite)
save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=True, DESI_style=False), 
                folder_to_save , '/cov_DM_DH.txt', overwrite=overwrite)

f = folder_to_save + '/redshifts.txt'
if overwrite:
    print("Overwriting file:", f)
    np.savetxt(f, z_centers, fmt="%.8e")
else:
    if not os.path.exists(f):
        np.savetxt(f, z_centers, fmt="%.8e")
    else:
        print("File already exists:", f)

File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky//cov_alpha.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky//cov_DM_DH.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky/redshifts.txt


In [91]:
data = create_DESI_fid_data(z_centers, DESI_style=False)
save_mean_data(data, folder_to_save, '/mean_LCDM_fid.txt', overwrite=overwrite)

File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky//mean_LCDM_fid.txt


In [92]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)
# save the summary.json file in the cov_mat folder
summary_file_path = folder_to_save + '/summary.json'
if overwrite:
    print("Overwriting file:", summary_file_path)
    with open(summary_file_path, 'w') as f:
        json.dump(summary, f, indent=4)
else:
    if not os.path.exists(summary_file_path):
        with open(summary_file_path, 'w') as f:
            json.dump(summary, f, indent=4)
    else:
        print("File already exists:", summary_file_path)

DESI style files

In [76]:
overwrite = False

In [ ]:
# Desi-style covariance matrix
# mask = [1, 1]
if z_centers[0] < 0.35:
    mask = True
    save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=True, DESI_style=mask), 
                        folder_to_save , '/cov_DM_DH_DESIstyle.txt', overwrite=overwrite)

    # if z_centers[0] < 0.35:
    data_DESI = create_DESI_fid_data(z_centers, DESI_style=mask)
    save_mean_data(data_DESI, folder_to_save, '/mean_LCDM_fid_DESIstyle.txt', overwrite=overwrite)

## combining forecast

In [37]:
def combine_cov_mat(cov1, cov2):
    """
    Combine two covariance matrices into a block diagonal matrix.
    
    Parameters:
    cov1 : np.ndarray
        First covariance matrix.
    cov2 : np.ndarray
        Second covariance matrix.
        
    Returns:
    np.ndarray
        Combined block diagonal covariance matrix.
    """
    if cov1.ndim != 2 or cov2.ndim != 2:
        raise ValueError("Both cov1 and cov2 must be 2D arrays.")

    return np.block([[cov1, np.zeros((cov1.shape[0], cov2.shape[1]))],
                     [np.zeros((cov2.shape[0], cov1.shape[1])), cov2]])

def combine_forecast(folder_list):
    '''
    Recursively combine the forecasts from multiple folders into a single one
    '''
    if len(folder_list) < 2:
        raise ValueError("At least two folders are required to combine forecasts.")
    elif len(folder_list) == 2:
        return combine(folder_list[0], folder_list[1])
    else:
        pair = combine_forecast(folder_list[1:])
        return combine(folder_list[0], pair)

def combine(folder1, folder2):
    #combine covariance matrices
    cov1 = np.loadtxt(folder1 + '/cov_DM_DH.txt')
    cov2 = np.loadtxt(folder2 + '/cov_DM_DH.txt')
    combined_cov_DM_DH = combine_cov_mat(cov1, cov2)
    cov1 = np.loadtxt(folder1 + '/cov_alpha.txt')
    cov2 = np.loadtxt(folder2 + '/cov_alpha.txt')
    combined_cov_alpha = combine_cov_mat(cov1, cov2)

    # DV/rd can only be in the first folder:
    combined_cov_DM_DH_DESIstyle = None
    if os.path.exists(folder1 + '/cov_DM_DH_DESIstyle.txt'):
        cov1 = np.loadtxt(folder1 + '/cov_DM_DH_DESIstyle.txt')
        cov2 = np.loadtxt(folder2 + '/cov_DM_DH.txt')
        combined_cov_DM_DH_DESIstyle = combine_cov_mat(cov1, cov2)

    # combine redshifts
    z1 = np.loadtxt(folder1 + '/redshifts.txt')
    if z1.ndim == 0: z1 = np.array([z1])
    z2 = np.loadtxt(folder2 + '/redshifts.txt')
    if z2.ndim == 0: z2 = np.array([z2])
    combined_z = np.concatenate((z1, z2))

    # combine mean data
    mean1 = pd.read_csv(
        folder1 + '/mean_LCDM_fid.txt',
        sep=r"\s+",
        header=None,
    )
    mean2 = pd.read_csv(
        folder2 + '/mean_LCDM_fid.txt',
        sep=r"\s+",
        header=None,
    )
    combined_mean = pd.concat([mean1, mean2], ignore_index=True)

    combined_mean_DESI = None
    if os.path.exists(folder1 + '/mean_LCDM_fid_DESIstyle.txt'):
        mean1_DESI = pd.read_csv(
            folder1 + '/mean_LCDM_fid_DESIstyle.txt',
            sep=r"\s+",
            header=None,
        )
        mean2_DESI = pd.read_csv(
            folder2 + '/mean_LCDM_fid.txt',
            sep=r"\s+",
            header=None,
        )
        combined_mean_DESI = pd.concat([mean1_DESI, mean2_DESI], ignore_index=True)

    return combined_cov_DM_DH, combined_cov_alpha, combined_cov_DM_DH_DESIstyle, combined_z, combined_mean, combined_mean_DESI

def remove_z(folder, redshifts):
    '''
    Remove given redshifts from the files.
    '''
    mean = pd.read_csv(folder + '/mean_LCDM_fid.txt', sep=r"\s+", header=None)
    z = mean.iloc[:,0].values
    z = np.array(z)
    redshifts = np.array(redshifts)
    
    mask = np.any(np.isclose(z[:, None], redshifts[None, :], atol=1e-8), axis=1)
    mask = ~mask

    new_mean = mean[mask]
    cov_DM_DH = np.loadtxt(folder + '/cov_DM_DH.txt')
    cov_alpha = np.loadtxt(folder + '/cov_alpha.txt')
    new_cov_DM_DH = cov_DM_DH[mask,:][:,mask]
    new_cov_alpha = cov_alpha[mask,:][:,mask]

    z = np.loadtxt(folder + '/redshifts.txt')
    if z.ndim == 0: z = np.array([z])
    mask = np.isin(z, redshifts, invert=True)
    new_z = z[mask]

    new_cov_DM_DH_DESIstyle = None
    new_mean_DESI_style = None
    if os.path.exists(folder + '/mean_LCDM_fid_DESIstyle.txt'):
        mean_DESI_style = pd.read_csv(folder + '/mean_LCDM_fid_DESIstyle.txt', sep=r"\s+", header=None)

        z = mean_DESI_style.iloc[:,0].values
        mask = np.isin(z, redshifts, invert=True)
        new_mean_DESI_style = mean_DESI_style[mask]
        cov_DM_DH_DESIstyle = np.loadtxt(folder + '/cov_DM_DH_DESIstyle.txt')
        new_cov_DM_DH_DESIstyle = cov_DM_DH_DESIstyle[mask,:][:,mask]

    return new_cov_DM_DH, new_cov_alpha, new_cov_DM_DH_DESIstyle, new_z, new_mean, new_mean_DESI_style

In [42]:
cDMDH, ca, c_DESI, z, m, m_DESI = remove_z('/home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_combined', [0.285])

In [ ]:
# temp_folder = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed'
# os.makedirs(temp_folder, exist_ok=True)
# save_a_cov_mat(cDMDH, temp_folder, 'cov_DM_DH.txt', overwrite=True)
# save_a_cov_mat(ca, temp_folder, 'cov_alpha.txt', overwrite=True)
# if c_DESI is not None:
#     save_a_cov_mat(c_DESI, temp_folder, 'cov_DM_DH_DESIstyle.txt', overwrite=True)
# save_mean_data(m, temp_folder, 'mean_LCDM_fid.txt', overwrite=True)
# if m_DESI is not None:
#     save_mean_data(m_DESI, temp_folder, 'mean_LCDM_fid_DESIstyle.txt', overwrite=True)
# np.savetxt(temp_folder + '/redshifts.txt', z, fmt="%.8e")

Overwriting file: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed/cov_DM_DH.txt
Overwriting file: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed/cov_alpha.txt
Overwriting file: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed/cov_DM_DH_DESIstyle.txt
Overwriting file: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed/mean_LCDM_fid.txt
Overwriting file: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/test_custombin_DESI_removed/mean_LCDM_fid_DESIstyle.txt


In [45]:
overwrite = False

In [93]:
fold1 = r'/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky'
fold2 = r'/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_QSO_fullsky'

cov_DM_DH, cov_alpha, cov_DM_DH_DESIstyle, combined_z, combined_mean, combined_mean_DESI = combine_forecast([fold1, fold2])

newfold = r'/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_fullsky_all'
if not os.path.exists(newfold):
    os.makedirs(newfold)

# save the combined covariance matrices and mean data
save_a_cov_mat(cov_DM_DH, newfold, '/cov_DM_DH.txt', overwrite=overwrite)
save_a_cov_mat(cov_alpha, newfold, '/cov_alpha.txt', overwrite=overwrite)
save_mean_data(combined_mean, newfold, '/mean_LCDM_fid.txt', overwrite=overwrite)
np.savetxt(newfold + '/redshifts.txt', combined_z, fmt="%.8e")

if cov_DM_DH_DESIstyle is not None:
    save_a_cov_mat(cov_DM_DH_DESIstyle, newfold, '/cov_DM_DH_DESIstyle.txt', overwrite=overwrite)

if combined_mean_DESI is not None:
    save_mean_data(combined_mean_DESI, newfold, '/mean_LCDM_fid_DESIstyle.txt', overwrite=overwrite)

In [58]:
print(pd.DataFrame(combined_mean))

        0          1           2
0   0.285   8.021697  DM_over_rs
1   0.285  26.004530  DH_over_rs
2   0.510  13.500537  DM_over_rs
3   0.510  22.739826  DH_over_rs
4   0.700  17.579508  DM_over_rs
5   0.700  20.243223  DH_over_rs
6   0.925  21.836231  DM_over_rs
7   0.925  17.663244  DH_over_rs
8   1.325  28.137752  DM_over_rs
9   1.325  14.033375  DH_over_rs
10  1.490  30.352565  DM_over_rs
11  1.490  12.838646  DH_over_rs
